# Clase 004 — Estructura reproducible de proyecto

**Parte 0 — Prerrequisitos** · cookiecutter-data-science v2.

> 🎯 Pasar de "carpeta con notebooks" a proyecto profesional con separación de datos, código y documentación.

> ⏱️ ~60 min

## 🗺️ La estructura estándar (CCDS v2)

```
mi-proyecto/
├── README.md
├── pyproject.toml          ← deps y metadata
├── Makefile                ← comandos del proyecto
├── data/
│   ├── raw/                ← INMUTABLE. Nunca editar.
│   ├── interim/            ← transformaciones intermedias
│   ├── processed/          ← listo para modelar
│   └── external/           ← datos de terceros
├── notebooks/
│   ├── 0.01-jvp-eda.ipynb  ← convención: <fase>.<n>-<iniciales>-<descripción>
│   └── 1.02-jvp-modelo.ipynb
├── src/
│   └── mi_proyecto/
│       ├── __init__.py
│       ├── data/           ← carga/limpieza
│       ├── features/       ← feature engineering
│       ├── models/         ← entrenamiento, predicción
│       └── visualization/  ← gráficos
├── reports/
│   └── figures/
├── tests/
└── docs/
```

No es dogma — es **convención**. La ventaja: cualquier DS que la conozca sabe dónde buscar.

## ⚙️ Por qué `data/raw` es sagrado

Regla: **nunca modifiques un archivo en `data/raw/`**. Todo procesamiento escribe a `data/interim/` o `data/processed/`.

**Por qué**:
- Si tu pipeline rompe, puedes regenerar todo desde el origen.
- Permite re-ejecutar análisis con datos diferentes (nuevo período, otra fuente).
- Hace explícito el grafo de dependencias (raw → interim → processed).

In [ ]:
# Demo: estructura típica creada con Path (simulación, no genera nada en disco)
from pathlib import Path

estructura = [
    'mi-proyecto/data/raw/',
    'mi-proyecto/data/interim/',
    'mi-proyecto/data/processed/',
    'mi-proyecto/notebooks/0.01-eda.ipynb',
    'mi-proyecto/src/mi_proyecto/__init__.py',
    'mi-proyecto/src/mi_proyecto/features.py',
    'mi-proyecto/pyproject.toml',
    'mi-proyecto/Makefile',
    'mi-proyecto/README.md',
]
for p in estructura:
    icon = '📁' if p.endswith('/') else '📄'
    print(f'{icon} {p}')

## 🐍 Notebooks vs `src/`

**Notebooks** = exploración. Bocetos. Análisis ad-hoc.
**`src/`** = código de producción que se reutiliza.

**Regla práctica**: cuando copias-pegas una función entre 2 notebooks, es momento de moverla a `src/`. Luego desde notebook:

```python
from mi_proyecto.features import limpia_fechas
df = limpia_fechas(df)
```

Para que esto funcione, el paquete debe estar **instalado en modo editable**:

```bash
pip install -e .   # desde la raíz del proyecto, con pyproject.toml
```

## 📦 `pyproject.toml` como fuente de verdad

En 2026, `pyproject.toml` es el estándar (PEP 621). Reemplaza `setup.py`, `setup.cfg`, y deja `requirements.txt` solo para lockfiles.

Ejemplo mínimo:

```toml
[project]
name = "mi-proyecto"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = [
    "numpy>=2.0",
    "pandas>=2.2",
    "matplotlib>=3.8",
    "scikit-learn>=1.4",
]

[project.optional-dependencies]
dev = ["pytest>=8", "ruff>=0.5", "mypy>=1.10"]

[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"
```

Luego: `pip install -e ".[dev]"` instala todo + tools de desarrollo.

## 🔧 `Makefile` como interfaz humana

```makefile
.PHONY: setup data train test clean

setup:
	python -m venv .venv && . .venv/bin/activate && pip install -e ".[dev]"

data:
	python -m mi_proyecto.data.make_dataset data/raw data/processed

train:
	python -m mi_proyecto.models.train

test:
	pytest tests/ -v

clean:
	rm -rf data/interim/* data/processed/* models/*
```

Ventaja: `make data` es más legible que recordar el comando exacto, y CI puede llamarlo igual que tú.

## 🚩 Olores de proyecto mal estructurado

Si ves alguno de estos, hay deuda técnica:

- `Untitled27.ipynb` ← notebook sin nombre = código que nadie va a leer
- `final_FINAL_v2.py` ← versionado a mano
- `data/customers_20240801_backup.csv` en git ← datos en git, fecha en filename
- 8 notebooks que cargan y limpian el CSV de la misma forma ← función no extraída
- `requirements.txt` con `numpy` (sin versión) ← reproducibilidad rota
- `notebook.ipynb` con celdas vacías y outputs gigantes ← `nbstripout` lo arregla

## ✅ Checklist

- [ ] Sé generar un proyecto con `cookiecutter-data-science`
- [ ] Entiendo por qué `data/raw/` no se modifica
- [ ] Sé importar desde `src/` en mis notebooks
- [ ] Mi proyecto tiene `pyproject.toml`, no `requirements.txt` suelto
- [ ] Reconozco al menos 3 olores de proyectos mal estructurados

## 📝 Homework

Ver `README.md`. Repo CCDS con `data/raw/penguins.csv`, notebook que importa de `src/`, `Makefile` con `make setup` y `make data`.

## 🔗 Referencias

- [cookiecutter-data-science v2](https://cookiecutter-data-science.drivendata.org/)
- Sculley et al., *Hidden Technical Debt in ML Systems* (NeurIPS 2015)

➡️ **Siguiente:** [005 — VS Code / Cursor para Python y Jupyter](../005-vs-code-cursor-para-python-y-jupyter/README.md)